In [6]:
import torch
import torchaudio
import os
import gc
import glob
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2FeatureExtractor, WavLMModel, HubertModel, Wav2Vec2Model

# ==============================================================================
# 1. DATASET TỐI ƯU
# ==============================================================================
class TestSpeakerDataset(Dataset):
    def __init__(self, folder_path):
        self.folder_path = folder_path
        # Chỉ lấy file .wav
        self.file_paths = glob.glob(os.path.join(folder_path, "**", "*.wav"), recursive=True)
        # Sắp xếp theo độ dài (ước tính qua dung lượng) giúp giảm padding thừa trong batch
        self.file_paths.sort(key=lambda x: os.path.getsize(x))
        
    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            waveform, sr = torchaudio.load(path)
            if sr != 16000:
                waveform = torchaudio.functional.resample(waveform, sr, 16000)
            
            # Chuyển mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0)
            else:
                waveform = waveform.squeeze(0)
                
            rel_path = os.path.relpath(path, self.folder_path)
            return waveform, rel_path
        except Exception as e:
            print(f"Lỗi file {path}: {e}")
            return torch.zeros(16000), "error"

def collate_fn_test(batch):
    # Lọc bỏ các file lỗi
    batch = [b for b in batch if b[1] != "error"]
    waveforms, rel_paths = zip(*batch)
    return list(waveforms), list(rel_paths)

# ==============================================================================
# 2. CẤU HÌNH CHO RTX 4060 (8GB)
# ==============================================================================
INPUT_FOLDER = r"D:\Study\7-SP26\DATxSLP\Test set O\test-O"
OUTPUT_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding"
MODEL_KEY = "wav2vec2" 
BATCH_SIZE = 16  # Có thể tăng lên 24-32 nếu file âm thanh ngắn (<10s)

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_map = {
    "wavlm": (WavLMModel, "microsoft/wavlm-base"),
    "hubert": (HubertModel, "facebook/hubert-base-ls960"),
    "wav2vec2": (Wav2Vec2Model, "facebook/wav2vec2-base-960h")
}

model_class, repo = model_map[MODEL_KEY]
processor = Wav2Vec2FeatureExtractor.from_pretrained(repo)

print(f"Loading {MODEL_KEY.upper()}...")
# Sử dụng torch_dtype=torch.float16 để tiết kiệm VRAM và tăng tốc trên 4060
model = model_class.from_pretrained(
    repo, 
    output_hidden_states=True,
    torch_dtype=torch.float16, 
    attn_implementation="eager" # Sử dụng Scaled Dot Product Attention (nhanh & nhẹ hơn eager)
).to(device).eval()

dataset = TestSpeakerDataset(INPUT_FOLDER)
# Với 32GB RAM DDR5, đặt num_workers=4 để load data nhanh hơn
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    collate_fn=collate_fn_test, 
    shuffle=False, 
    num_workers=0, 
    pin_memory=True
)

all_embeddings_dict = {}

# ==============================================================================
# 3. TRÍCH XUẤT (SỬ DỤNG FP16)
# ==============================================================================
print(f"\n--- TRÍCH XUẤT TRÊN {device.upper()} (FP16 MODE) ---")

with torch.inference_mode():
    for waveforms, rel_paths in tqdm(dataloader, desc="Processing"):
        
        # SỬA TẠI ĐÂY: Chuyển list các tensor thành list các numpy array 1D
        # Điều này giúp processor nhận diện đúng cấu trúc để thực hiện padding
        waveforms_input = [w.numpy() for w in waveforms]

        try:
            # Thực hiện padding và chuyển sang tensor
            inputs = processor(
                waveforms_input, 
                sampling_rate=16000, 
                return_tensors="pt", 
                padding=True
            ).to(device)
            
            # Chuyển sang FP16 để tối ưu VRAM cho 4060
            inputs = {k: v.to(torch.float16) if torch.is_floating_point(v) else v for k, v in inputs.items()}

            outputs = model(**inputs)
            
            # Trích xuất hidden states
            # [Layers, Batch, Time, Dim]
            stacked = torch.stack(outputs.hidden_states) 
            
            # Mean pooling theo chiều Time (dim=2) -> [Layers, Batch, Dim]
            # Sau đó permute về [Batch, Layers, Dim]
            pooled = stacked.mean(dim=2).permute(1, 0, 2).cpu()
            
            for j in range(len(rel_paths)):
                all_embeddings_dict[rel_paths[j]] = pooled[j]

        except torch.cuda.OutOfMemoryError:
            # Nếu vẫn OOM (do batch có file quá dài), giải phóng cache và chạy đơn lẻ
            torch.cuda.empty_cache()
            for w, p_path in zip(waveforms_input, rel_paths):
                # Xử lý đơn lẻ 1 file
                inp = processor(w, sampling_rate=16000, return_tensors="pt").to(device)
                inp = {k: v.to(torch.float16) if torch.is_floating_point(v) else v for k, v in inp.items()}
                
                out = model(**inp)
                p = torch.stack(out.hidden_states).mean(dim=2).permute(1, 0, 2).cpu()
                all_embeddings_dict[p_path] = p[0]
            torch.cuda.empty_cache()

# Lưu kết quả
save_file = os.path.join(OUTPUT_DIR, f"all_embeddings_{MODEL_KEY}.pt")
torch.save(all_embeddings_dict, save_file)

# Clear bộ nhớ
del model, processor
gc.collect()
torch.cuda.empty_cache()

print(f"✅ Xong! Đã lưu {len(all_embeddings_dict)} file vào: {save_file}")

c:\Users\Lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading WAV2VEC2...


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- TRÍCH XUẤT TRÊN CUDA (FP16 MODE) ---


Processing: 100%|██████████| 1112/1112 [06:05<00:00,  3.04it/s]


✅ Xong! Đã lưu 17786 file vào: D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding\all_embeddings_wav2vec2.pt


In [7]:
import torch

# Thay 'đường_dẫn_tới_file.pt' bằng tên file thực tế của bạn
file_path = r'D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding\all_embeddings_wav2vec2.pt'

try:
    # Load file .pt (sử dụng map_location='cpu' để tránh lỗi nếu file được lưu trên GPU nhưng máy bạn chỉ có CPU)
    data = torch.load(file_path, map_location='cpu')

    # Kiểm tra xem dữ liệu load lên có phải là dictionary hay không
    if isinstance(data, dict):
        dict_length = len(data)
        print(f"✅ Độ dài của dictionary là: {dict_length}")
        
        # Tùy chọn: In ra một số keys đầu tiên để dễ hình dung
        # print(f"Các keys có trong dict: {list(data.keys())[:5]}")
    else:
        print(f"⚠️ File không chứa dictionary ở cấp cao nhất. Kiểu dữ liệu thực tế: {type(data)}")

except Exception as e:
    print(f"❌ Có lỗi xảy ra khi đọc file: {e}")

✅ Độ dài của dictionary là: 17786
